# Train the cloud-removal model on Google Colab

This notebook trains the Swin U-Net on **RICE1**, measures it on held-out test images, and saves everything you need for the website.

**Before you start:** `Runtime → Change runtime type → T4 GPU`. Training takes roughly **1–1.5 hours** on a T4.

If Colab disconnects, just run all cells again: training resumes from the last finished epoch because checkpoints are saved to your Google Drive.

## 1. Check the GPU
Why: training on CPU would take days. This should print a line like `GPU 0: Tesla T4`.

In [ ]:
!nvidia-smi -L

## 2. Connect Google Drive
Why: Colab deletes everything in `/content` when the session ends. Checkpoints saved in Drive survive, so a disconnect costs you at most one epoch.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

RUN_DIR = '/content/drive/MyDrive/cloud-removal/runs/rice1'   # checkpoints + logs live here

## 3. Get the code
Why: the notebook only *runs* the project; all the real code lives in the GitHub repo, so there is one copy of it. Change `REPO_URL` if your repo has a different name.

In [ ]:
import os

REPO_URL = 'https://github.com/aaryanrana01/cloud-removal.git'
REPO_DIR = '/content/cloud-removal'

if os.path.exists(REPO_DIR):
    !git -C {REPO_DIR} pull --ff-only
else:
    !git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!pip install -q -r requirements-train.txt

## 4. Download RICE1 from Kaggle
Why: the dataset is not stored in the repo (it is large and not ours to redistribute).

You need `kaggle.json` (Kaggle → Settings → *Create New Token*). Put it in the top of your Google Drive once (`MyDrive/kaggle.json`) and you will never be asked again; otherwise this cell asks you to upload it.

In [ ]:
import os
import shutil
import zipfile
from pathlib import Path

DATA_DIR = '/content/data'
os.makedirs('/root/.kaggle', exist_ok=True)
drive_key = '/content/drive/MyDrive/kaggle.json'
if os.path.exists(drive_key):
    shutil.copy(drive_key, '/root/.kaggle/kaggle.json')
elif not os.path.exists('/root/.kaggle/kaggle.json'):
    from google.colab import files
    print('Upload kaggle.json'); up = files.upload()
    Path('/root/.kaggle/kaggle.json').write_bytes(list(up.values())[0])
os.chmod('/root/.kaggle/kaggle.json', 0o600)

if not os.path.exists(DATA_DIR):
    !kaggle datasets download -d shubhank001/rice-remote-sensing-images-for-cloud-removal -p /content/
    with zipfile.ZipFile('/content/rice-remote-sensing-images-for-cloud-removal.zip') as z:
        z.extractall(DATA_DIR)

from cloudremoval.data import collect_pairs, find_dataset_root

root = find_dataset_root(DATA_DIR)
print('RICE1 folder:', root, '| pairs:', len(collect_pairs(root)))   # expect 500

## 5. Train
Why each setting is what it is: see `configs/rice1.yaml` (every number is commented there).

What you will see: one line per epoch with validation **PSNR** (higher = closer to the real clear image). It should climb quickly for the first ~10 epochs, then slowly. The best model is saved as `best.pt` whenever validation PSNR improves.

In [ ]:
!python scripts/train.py --config configs/rice1.yaml --data-root {DATA_DIR} --out-dir {RUN_DIR}

## 6. Look at the training curves and a preview
Each preview row is: **cloudy input | model output | real clear image** (validation images the model never trained on).

In [ ]:
import glob

from IPython.display import Image as Show
from IPython.display import display

display(Show(f'{RUN_DIR}/training_curves.png'))
display(Show(sorted(glob.glob(f'{RUN_DIR}/previews/*.png'))[-1], width=900))

## 7. Evaluate on the 100 test images
Why: validation images were used to pick the best epoch, so they are slightly 'seen'. The **test** images were never used for anything. These are the numbers you put on your resume and on the website.

It also scores the cloudy input itself (**No processing**). The model is only useful if it clearly beats that.

In [ ]:
!python scripts/evaluate.py --checkpoint {RUN_DIR}/best.pt --data-root {DATA_DIR} --split {RUN_DIR}/splits.json
from IPython.display import Markdown

display(Markdown(open('results/results.md').read()))
for p in sorted(glob.glob('results/examples/*.png'))[:4]:
    display(Show(p, width=900))

## 8. Download the results for the GitHub repo
Why: the website shows the real test numbers (`results/metrics.json`) and a gallery of test images (`server/samples/`). These files are small, so they go in the GitHub repo. The model weights do **not** (step 9).

This downloads `results_bundle.zip`. Unzip it into your repo folder (it has the same folder layout) and commit.

In [ ]:
!cp {RUN_DIR}/splits.json results/splits.json
!cp {RUN_DIR}/training_curves.png results/training_curves.png
!zip -qr /content/results_bundle.zip results server/samples
!cp /content/results_bundle.zip {RUN_DIR}/
from google.colab import files

files.download('/content/results_bundle.zip')

## 9. Upload the model to Hugging Face
Why: GitHub rejects files over 100 MB and is the wrong place for model weights. A Hugging Face **model repo** stores them, shows a model card, and the website downloads the weights from there when it starts.

1. Create a free account at huggingface.co and a token with **write** access (Settings → Access Tokens).
2. Set `HF_USERNAME` below and run the cell; paste the token when asked.

In [ ]:
HF_USERNAME = 'your-hf-username'          # <- change this
MODEL_REPO = f'{HF_USERNAME}/cloud-removal-swin'

import json

from huggingface_hub import create_repo, login, upload_file

login()
create_repo(MODEL_REPO, repo_type='model', exist_ok=True)

m = json.load(open('results/metrics.json'))
card = f'''---
license: mit
tags: [image-to-image, cloud-removal, remote-sensing, swin-transformer]
---
# Swin U-Net for satellite cloud removal (RICE1)

{m['parameters']:,} parameters. Test set: {m['test_images']} held-out RICE1 images ({m['image_size'][0]}x{m['image_size'][1]}).

| | PSNR (dB) | SSIM | CC |
|---|---|---|---|
| No processing | {m['no_processing']['psnr']['mean']:.2f} | {m['no_processing']['ssim']['mean']:.4f} | {m['no_processing']['cc']['mean']:.4f} |
| This model | {m['model']['psnr']['mean']:.2f} | {m['model']['ssim']['mean']:.4f} | {m['model']['cc']['mean']:.4f} |

Load with `cloudremoval.inference.Predictor.from_checkpoint('best.pt')` from the project repo.
'''
open('/content/MODEL_CARD.md', 'w').write(card)
upload_file(path_or_fileobj=f'{RUN_DIR}/best.pt', path_in_repo='best.pt', repo_id=MODEL_REPO)
upload_file(path_or_fileobj='/content/MODEL_CARD.md', path_in_repo='README.md', repo_id=MODEL_REPO)
print('Done ->', f'https://huggingface.co/{MODEL_REPO}')